In [7]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("financial_rag")

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

anthropic_key = os.getenv("ANTHROPIC_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

In [9]:
import fitz  # pymupdf
import json
import os

# map filename keywords to bank names
BANK_NAMES = {
    "hdfc": "HDFC Bank",
    "icici": "ICICI Bank",
    "sbi": "SBI",
    "axis": "Axis Bank",
    "kotak": "Kotak Mahindra Bank"
}

def get_bank_name(filename):
    filename_lower = filename.lower()
    for keyword, bank_name in BANK_NAMES.items():
        if keyword in filename_lower:
            return bank_name
    return "Unknown"

def parse_pdfs(pdf_folder, output_path):
    all_pages = []
    
    for filename in os.listdir(pdf_folder):
        if not filename.endswith(".pdf"):
            continue
            
        bank_name = get_bank_name(filename)
        pdf_path = os.path.join(pdf_folder, filename)
        
        print(f"Parsing {bank_name}...")
        
        doc = fitz.open(pdf_path)
        
        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()
            
            # skip blank or near-blank pages
            if len(text.strip()) < 50:
                continue
            
            all_pages.append({
                "bank_name": bank_name,
                "page_number": page_num,
                "fiscal_year": "FY25",
                "text": text
            })
        
        print(f"  → {len(doc)} pages processed")
        doc.close()
    
    # save to json
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_pages, f, ensure_ascii=False, indent=2)
    
    print(f"\nDone. {len(all_pages)} pages saved to {output_path}")

# run it
parse_pdfs("data/pdfs", "data/parsed_pages.json")

Parsing Axis Bank...
  → 504 pages processed
Parsing HDFC Bank...
  → 590 pages processed
Parsing ICICI Bank...
  → 341 pages processed
Parsing Kotak Mahindra Bank...
  → 522 pages processed
Parsing SBI...
  → 517 pages processed

Done. 2460 pages saved to data/parsed_pages.json


In [10]:
# peek at one page to make sure data looks right
with open("data/parsed_pages.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

# print first page
print(pages[0])

{'bank_name': 'Axis Bank', 'page_number': 2, 'fiscal_year': 'FY25', 'text': 'Inside this report\nReporting context\n4\t\nAbout the report\nWelcome to Axis Bank\n10\t\nAbout Axis Bank\n12\t Integrated business lines\n14\t\nChairman’s statement\n16\t\nStrategic pillars\n18\t\nAdvancing our ESG agenda\n20\t Presence\n22\t Milestones\n24\t\nOwnership structure\n26 \nOperating landscape\n30 Marketing Initiatives\n40\t Board of directors\n41\t\nCore management team\nValue creation by the bank\n44 Value creation model\n48\t Stakeholder engagement\n58\t Materiality assessment\nBusiness performance review\n70\t MD & CEO’s statement \n76\t\nExternal environment  \n84 Strategy in action\n88\t Key performance indicators  \n92\t Message from the management – Retail banking  \n96\t\nBusiness segment performance – Retail banking  \n102\t Message from the management – Wholesale banking \n104\t Business segment performance – Wholesale banking  \n108 ‘One Axis’ in action \n116 \x07Message from the manag

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

def chunk_pages(input_path, output_path):
    # load parsed pages
    with open(input_path, "r", encoding="utf-8") as f:
        pages = json.load(f)
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        length_function=len,
    )
    
    all_chunks = []
    chunk_id = 0
    
    for page in pages:
        # skip if text too short
        if len(page["text"].strip()) < 50:
            continue
        
        chunks = splitter.split_text(page["text"])
        
        for chunk in chunks:
            all_chunks.append({
                "chunk_id": f"chunk_{chunk_id}",
                "bank_name": page["bank_name"],
                "page_number": page["page_number"],
                "fiscal_year": page["fiscal_year"],
                "text": chunk,
                "level": "leaf"
            })
            chunk_id += 1
    
    # save to json
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)
    
    print(f"Done. {len(all_chunks)} chunks saved to {output_path}")

# run it
chunk_pages("data/parsed_pages.json", "data/chunked_data.json")

Done. 11579 chunks saved to data/chunked_data.json


In [13]:
# peek at one chunk
with open("data/chunked_data.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0])

Total chunks: 11579

Sample chunk:
{'chunk_id': 'chunk_0', 'bank_name': 'Axis Bank', 'page_number': 2, 'fiscal_year': 'FY25', 'text': 'Inside this report\nReporting context\n4\t\nAbout the report\nWelcome to Axis Bank\n10\t\nAbout Axis Bank\n12\t Integrated business lines\n14\t\nChairman’s statement\n16\t\nStrategic pillars\n18\t\nAdvancing our ESG agenda\n20\t Presence\n22\t Milestones\n24\t\nOwnership structure\n26 \nOperating landscape\n30 Marketing Initiatives\n40\t Board of directors\n41\t\nCore management team\nValue creation by the bank\n44 Value creation model\n48\t Stakeholder engagement\n58\t Materiality assessment\nBusiness performance review\n70\t MD & CEO’s statement \n76\t\nExternal environment  \n84 Strategy in action\n88\t Key performance indicators  \n92\t Message from the management – Retail banking  \n96\t\nBusiness segment performance – Retail banking  \n102\t Message from the management – Wholesale banking', 'level': 'leaf'}


In [18]:
from openai import OpenAI
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()

client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def embed_texts_with_retry(texts, batch_size=50, max_retries=3):
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        for attempt in range(max_retries):
            try:
                response = client_openai.embeddings.create(
                    input=batch,
                    model="text-embedding-3-small"
                )
                batch_embeddings = [item.embedding for item in response.data]
                all_embeddings.extend(batch_embeddings)
                print(f"  Embedded {min(i+batch_size, len(texts))}/{len(texts)} chunks...")
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"  Error on batch {i} — retrying in 5 seconds... ({e})")
                    time.sleep(5)
                else:
                    print(f"  Failed after {max_retries} attempts. Saving progress...")
                    # save whatever we have so far
                    for j, chunk in enumerate(chunks[:len(all_embeddings)]):
                        chunk["embedding"] = all_embeddings[j]
                    with open("data/chunks_with_embeddings.json", "w", encoding="utf-8") as f:
                        json.dump(chunks[:len(all_embeddings)], f, ensure_ascii=False, indent=2)
                    print(f"  Saved {len(all_embeddings)} chunks. Re-run to continue.")
                    raise
    
    return all_embeddings

# load chunks
with open("data/chunked_data.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# check if we already have partial embeddings
already_done = 0
if os.path.exists("data/chunks_with_embeddings.json"):
    with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
        existing = json.load(f)
    already_done = len(existing)
    print(f"Resuming from chunk {already_done}...")
    # copy existing embeddings back
    for i in range(already_done):
        chunks[i]["embedding"] = existing[i]["embedding"]

# only embed remaining chunks
remaining_texts = [chunk["text"] for chunk in chunks[already_done:]]

if len(remaining_texts) == 0:
    print("All chunks already embedded!")
else:
    print(f"Embedding {len(remaining_texts)} remaining chunks...")
    new_embeddings = embed_texts_with_retry(remaining_texts)
    
    for i, embedding in enumerate(new_embeddings):
        chunks[already_done + i]["embedding"] = embedding

    with open("data/chunks_with_embeddings.json", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"\nDone. All {len(chunks)} chunks embedded and saved.")

Resuming from chunk 0...
Embedding 11579 remaining chunks...
  Embedded 50/11579 chunks...
  Embedded 100/11579 chunks...
  Embedded 150/11579 chunks...
  Embedded 200/11579 chunks...
  Embedded 250/11579 chunks...
  Embedded 300/11579 chunks...
  Embedded 350/11579 chunks...
  Embedded 400/11579 chunks...
  Embedded 450/11579 chunks...
  Embedded 500/11579 chunks...
  Embedded 550/11579 chunks...
  Embedded 600/11579 chunks...
  Embedded 650/11579 chunks...
  Embedded 700/11579 chunks...
  Embedded 750/11579 chunks...
  Embedded 800/11579 chunks...
  Embedded 850/11579 chunks...
  Embedded 900/11579 chunks...
  Embedded 950/11579 chunks...
  Embedded 1000/11579 chunks...
  Embedded 1050/11579 chunks...
  Embedded 1100/11579 chunks...
  Embedded 1150/11579 chunks...
  Embedded 1200/11579 chunks...
  Embedded 1250/11579 chunks...
  Embedded 1300/11579 chunks...
  Embedded 1350/11579 chunks...
  Embedded 1400/11579 chunks...
  Embedded 1450/11579 chunks...
  Embedded 1500/11579 chunks...

In [19]:
# run this to check exact token count
import json
import tiktoken

encoder = tiktoken.encoding_for_model("text-embedding-3-small")

with open("data/chunked_data.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

total_tokens = sum(len(encoder.encode(chunk["text"])) for chunk in chunks)

print(f"Total tokens: {total_tokens:,}")
print(f"Estimated cost: ${total_tokens / 1_000_000 * 0.02:.4f}")

Total tokens: 2,000,508
Estimated cost: $0.0400


In [20]:
import chromadb
import json

# connect to chromadb
client_chroma = chromadb.PersistentClient(path="./chroma_db")
collection = client_chroma.get_or_create_collection(
    name="financial_rag",
    metadata={"hnsw:space": "cosine"}
)

# load chunks with embeddings
with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# upload in batches of 100
batch_size = 100
total = len(chunks)

for i in range(0, total, batch_size):
    batch = chunks[i:i+batch_size]
    
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        embeddings=[c["embedding"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{
            "bank_name": c["bank_name"],
            "page_number": c["page_number"],
            "fiscal_year": c["fiscal_year"],
            "level": c["level"]
        } for c in batch]
    )
    
    print(f"Uploaded {min(i+batch_size, total)}/{total} chunks...")

print(f"\nDone. {collection.count()} chunks in ChromaDB.")

Uploaded 100/11579 chunks...
Uploaded 200/11579 chunks...
Uploaded 300/11579 chunks...
Uploaded 400/11579 chunks...
Uploaded 500/11579 chunks...
Uploaded 600/11579 chunks...
Uploaded 700/11579 chunks...
Uploaded 800/11579 chunks...
Uploaded 900/11579 chunks...
Uploaded 1000/11579 chunks...
Uploaded 1100/11579 chunks...
Uploaded 1200/11579 chunks...
Uploaded 1300/11579 chunks...
Uploaded 1400/11579 chunks...
Uploaded 1500/11579 chunks...
Uploaded 1600/11579 chunks...
Uploaded 1700/11579 chunks...
Uploaded 1800/11579 chunks...
Uploaded 1900/11579 chunks...
Uploaded 2000/11579 chunks...
Uploaded 2100/11579 chunks...
Uploaded 2200/11579 chunks...
Uploaded 2300/11579 chunks...
Uploaded 2400/11579 chunks...
Uploaded 2500/11579 chunks...
Uploaded 2600/11579 chunks...
Uploaded 2700/11579 chunks...
Uploaded 2800/11579 chunks...
Uploaded 2900/11579 chunks...
Uploaded 3000/11579 chunks...
Uploaded 3100/11579 chunks...
Uploaded 3200/11579 chunks...
Uploaded 3300/11579 chunks...
Uploaded 3400/11579

In [21]:
# test a simple query
query = "What is the NPA ratio of HDFC Bank?"

# embed the query
query_embedding = client_openai.embeddings.create(
    input=query,
    model="text-embedding-3-small"
).data[0].embedding

# search chromadb
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# print results
for i in range(3):
    print(f"\n--- Result {i+1} ---")
    print(f"Bank: {results['metadatas'][0][i]['bank_name']}")
    print(f"Page: {results['metadatas'][0][i]['page_number']}")
    print(f"Distance: {results['distances'][0][i]:.4f}")
    print(f"Text: {results['documents'][0][i][:200]}...")


--- Result 1 ---
Bank: HDFC Bank
Page: 38
Distance: 0.6798
Text: This was achieved through the extensive reach 
across 9,455 branches as well as a strong digital 
footprint. Our Proﬁt After Tax grew by 10.7 per cent 
and GNPA stood at 1.33 per cent. 
The Cost to In...

--- Result 2 ---
Bank: HDFC Bank
Page: 38
Distance: 0.6868
Text: ROE of 14.6 per cent. The Earnings Per Share (EPS) 
increased by 2.9 per cent to J88.3, while dividend 
per share rose by 12.8 per cent to J22.0 in FY25.
23,79,786 
18,83,395 
27,14,715 
FY24
FY25
FY2...

--- Result 3 ---
Bank: HDFC Bank
Page: 361
Distance: 0.7391
Text: HDFC Bank Limited
366
SCHEDULES TO THE STANDALONE FINANCIAL STATEMENTS
For the year ended March 31, 2025
 
^  NPAs represents advances aggregating to ` 35,194.95 crore (previous year: ` 31,056.65 cror...


In [22]:
from sklearn.cluster import KMeans
import numpy as np
import json

# load chunks with embeddings
with open("data/chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# group chunks by bank
banks = {}
for chunk in chunks:
    bank = chunk["bank_name"]
    if bank not in banks:
        banks[bank] = []
    banks[bank].append(chunk)

# print count per bank
for bank, bank_chunks in banks.items():
    print(f"{bank}: {len(bank_chunks)} chunks")

Axis Bank: 2182 chunks
HDFC Bank: 2672 chunks
ICICI Bank: 1550 chunks
Kotak Mahindra Bank: 2546 chunks
SBI: 2629 chunks


In [31]:
import anthropic
import numpy as np
from sklearn.cluster import KMeans
import json
import os
from dotenv import load_dotenv

load_dotenv()

client_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def cluster_and_summarise(bank_name, bank_chunks, n_clusters=5):
    print(f"\nProcessing {bank_name}...")
    
    # get embeddings as matrix
    embeddings_matrix = np.array([c["embedding"] for c in bank_chunks])
    
    # cluster
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings_matrix)
    
    l1_summaries = []
    
    for cluster_id in range(n_clusters):
        # get all chunks in this cluster
        cluster_chunks = [bank_chunks[i] for i in range(len(bank_chunks)) if labels[i] == cluster_id]
        
        # join texts (limit to avoid token limits)
        combined_text = "\n\n".join([c["text"] for c in cluster_chunks[:20]])
        
        print(f"  Cluster {cluster_id+1}: {len(cluster_chunks)} chunks → summarising...")
        
        # summarise with claude
        response = client_anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1000,
            messages=[{
                "role": "user",
                "content": f"""You are summarising a cluster of excerpts from {bank_name}'s FY25 annual report.
Write a 250-300 word summary covering the key financial metrics, risk factors, and strategic points in these excerpts. Be specific with numbers.

Context:
{combined_text}

Write only the summary, no preamble."""
            }]
        )
        
        summary_text = response.content[0].text
        
        l1_summaries.append({
            "chunk_id": f"l1_{bank_name.replace(' ', '_')}_{cluster_id}",
            "bank_name": bank_name,
            "cluster_id": cluster_id,
            "fiscal_year": "FY25",
            "text": summary_text,
            "level": "summary_l1",
            "source_chunk_count": len(cluster_chunks)
        })
    
    return l1_summaries

# run for all banks
all_l1_summaries = []

for bank_name, bank_chunks in banks.items():
    l1 = cluster_and_summarise(bank_name, bank_chunks)
    all_l1_summaries.extend(l1)

# save
with open("data/l1_summaries.json", "w", encoding="utf-8") as f:
    json.dump(all_l1_summaries, f, ensure_ascii=False, indent=2)

print(f"\nDone. {len(all_l1_summaries)} L1 summaries saved.")


Processing Axis Bank...
  Cluster 1: 511 chunks → summarising...
  Cluster 2: 473 chunks → summarising...
  Cluster 3: 357 chunks → summarising...
  Cluster 4: 308 chunks → summarising...
  Cluster 5: 533 chunks → summarising...

Processing HDFC Bank...
  Cluster 1: 627 chunks → summarising...
  Cluster 2: 590 chunks → summarising...
  Cluster 3: 411 chunks → summarising...
  Cluster 4: 669 chunks → summarising...
  Cluster 5: 375 chunks → summarising...

Processing ICICI Bank...
  Cluster 1: 285 chunks → summarising...
  Cluster 2: 239 chunks → summarising...
  Cluster 3: 360 chunks → summarising...
  Cluster 4: 340 chunks → summarising...
  Cluster 5: 326 chunks → summarising...

Processing Kotak Mahindra Bank...
  Cluster 1: 760 chunks → summarising...
  Cluster 2: 429 chunks → summarising...
  Cluster 3: 569 chunks → summarising...
  Cluster 4: 447 chunks → summarising...
  Cluster 5: 341 chunks → summarising...

Processing SBI...
  Cluster 1: 479 chunks → summarising...
  Cluster

In [32]:
def generate_l2_summaries(l1_summaries):
    # group l1 summaries by bank
    banks_l1 = {}
    for summary in l1_summaries:
        bank = summary["bank_name"]
        if bank not in banks_l1:
            banks_l1[bank] = []
        banks_l1[bank].append(summary)
    
    l2_summaries = []
    
    for bank_name, bank_l1s in banks_l1.items():
        print(f"Generating L2 summary for {bank_name}...")
        
        # combine all l1 summaries for this bank
        combined = "\n\n".join([s["text"] for s in bank_l1s])
        
        response = client_anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1500,
            messages=[{
                "role": "user",
                "content": f"""You are creating a comprehensive summary of {bank_name}'s FY25 annual report.
Below are 5 cluster summaries covering different aspects of the report.
Write a 400-500 word master summary covering the bank's overall financial performance, key metrics, risks, and strategy. Be specific with numbers.

Cluster summaries:
{combined}

Write only the summary, no preamble."""
            }]
        )
        
        l2_summaries.append({
            "chunk_id": f"l2_{bank_name.replace(' ', '_')}",
            "bank_name": bank_name,
            "fiscal_year": "FY25",
            "text": response.content[0].text,
            "level": "summary_l2"
        })
        
        print(f"  → Done")
    
    return l2_summaries

# run it
with open("data/l1_summaries.json", "r", encoding="utf-8") as f:
    l1_summaries = json.load(f)

l2_summaries = generate_l2_summaries(l1_summaries)

with open("data/l2_summaries.json", "w", encoding="utf-8") as f:
    json.dump(l2_summaries, f, ensure_ascii=False, indent=2)

print(f"\nDone. {len(l2_summaries)} L2 summaries saved.")

Generating L2 summary for Axis Bank...
  → Done
Generating L2 summary for HDFC Bank...
  → Done
Generating L2 summary for ICICI Bank...
  → Done
Generating L2 summary for Kotak Mahindra Bank...
  → Done
Generating L2 summary for SBI...
  → Done

Done. 5 L2 summaries saved.


In [33]:
# load all summaries
with open("data/l1_summaries.json", "r", encoding="utf-8") as f:
    l1_summaries = json.load(f)

with open("data/l2_summaries.json", "r", encoding="utf-8") as f:
    l2_summaries = json.load(f)

all_summaries = l1_summaries + l2_summaries
print(f"Total summaries to embed: {len(all_summaries)}")

# embed all summaries
summary_texts = [s["text"] for s in all_summaries]
summary_embeddings = embed_texts_with_retry(summary_texts, batch_size=50)

# upload to chromadb
for i, summary in enumerate(all_summaries):
    summary["embedding"] = summary_embeddings[i]

collection.add(
    ids=[s["chunk_id"] for s in all_summaries],
    embeddings=[s["embedding"] for s in all_summaries],
    documents=[s["text"] for s in all_summaries],
    metadatas=[{
        "bank_name": s["bank_name"],
        "fiscal_year": s["fiscal_year"],
        "level": s["level"]
    } for s in all_summaries]
)

print(f"\nDone. ChromaDB now has {collection.count()} total nodes.")

Total summaries to embed: 30
  Embedded 30/30 chunks...

Done. ChromaDB now has 11609 total nodes.


In [ ]:
def classify_query(question):
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=10,
        messages=[{
            "role": "user",
            "content": f"""Classify the following question into exactly one of these categories:
factual - single metric or fact from one specific bank
comparative - same metric or topic across multiple banks
summary - broad qualitative question about a bank's approach or strategy
out_of_scope - completely unrelated to banking and finance 
              (e.g. sports, weather, cooking)
              NOT out_of_scope: anything that could appear in a bank's annual report
              including leadership, board, strategy, operations, ESG

Respond with one word only: factual, comparative, summary, or out_of_scope

Question: {question}"""
        }]
    )
    
    return response.content[0].text.strip().lower()

# test all 4 query types
test_queries = [
    "What was HDFC Bank's gross NPA ratio in FY25?",
    "Compare capital adequacy ratios across all 5 banks.",
    "Summarise ICICI Bank's risk management approach.",
    "What is the RBI repo rate?",
    "What were SBI's net interest margins?",
    "Which bank had the highest return on equity?",
]

print("Testing router...\n")
for query in test_queries:
    category = classify_query(query)
    print(f"Q: {query}")
    print(f"→ {category}\n")

Testing router...

Q: What was HDFC Bank's gross NPA ratio in FY25?
→ factual

Q: Compare capital adequacy ratios across all 5 banks.
→ comparative

Q: Summarise ICICI Bank's risk management approach.
→ summary

Q: What is the RBI repo rate?
→ out_of_scope

Q: What were SBI's net interest margins?
→ factual

Q: Which bank had the highest return on equity?
→ comparative



In [ ]:
# cell 1 - imports and clients 
# to reinitailse
import anthropic
import chromadb
from openai import OpenAI
import os
import json
from dotenv import load_dotenv

load_dotenv(override=True)

client_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# reconnect to chromadb
client_chroma = chromadb.PersistentClient(path="./chroma_db")
collection = client_chroma.get_or_create_collection(
    name="financial_rag",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB has {collection.count()} nodes")
print("Clients ready ✓")

ChromaDB has 11609 nodes
Clients ready ✓


In [ ]:
def embed_text(text):
    response = client_openai.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def hyde_retrieve(question, n_results=5):
    # generate hypothetical answer
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""Write a one-paragraph excerpt from an Indian bank annual report 
that would answer this question: {question}
Use specific financial numbers and banking terminology.
Respond with only the paragraph, no preamble."""
        }]
    )
    
    hypothetical_answer = response.content[0].text
    print(f"  Hypothetical answer: {hypothetical_answer[:100]}...")
    
    # embed the hypothetical answer not the question
    hyde_embedding = embed_text(hypothetical_answer)
    
    # retrieve using hypothetical embedding
    results = collection.query(
        query_embeddings=[hyde_embedding],
        n_results=n_results,
        where={"level": "leaf"},
        include=["documents", "metadatas", "distances"]
    )
    
    return results

def multi_query_retrieve(question, n_results=5):
    # generate 3 paraphrases
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""Generate 3 different phrasings of this question for document retrieval:
{question}
Each phrasing should use different terminology to maximise retrieval coverage.
Return as a numbered list. No explanation."""
        }]
    )
    
    paraphrases_text = response.content[0].text
    paraphrases = [line.strip().lstrip("123.") .strip() 
                   for line in paraphrases_text.strip().split("\n") 
                   if line.strip()]
    
    print(f"  Paraphrases generated: {len(paraphrases)}")
    
    # retrieve for each paraphrase
    all_chunks = {}
    
    for paraphrase in paraphrases:
        embedding = embed_text(paraphrase)
        results = collection.query(
            query_embeddings=[embedding],
            n_results=n_results,
            where={"level": {"$in": ["leaf", "summary_l1", "summary_l2"]}},
            include=["documents", "metadatas", "distances"]
        )
        
        # deduplicate by document text
        for i in range(len(results["documents"][0])):
            doc = results["documents"][0][i]
            meta = results["metadatas"][0][i]
            dist = results["distances"][0][i]
            chunk_key = doc[:100]  # use first 100 chars as key
            if chunk_key not in all_chunks:
                all_chunks[chunk_key] = {
                    "text": doc,
                    "metadata": meta,
                    "distance": dist
                }
    
    print(f"  Unique chunks after deduplication: {len(all_chunks)}")
    return list(all_chunks.values())

def summary_retrieve(question, bank_name=None):
    # retrieve l2 summary nodes
    where_filter = {"level": "summary_l2"}
    if bank_name:
        where_filter = {"$and": [{"level": "summary_l2"}, {"bank_name": bank_name}]}
    
    embedding = embed_text(question)
    results = collection.query(
        query_embeddings=[embedding],
        n_results=5,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )
    
    return results

# test all three
print("=== Testing HyDE (factual) ===")
hyde_results = hyde_retrieve("What was HDFC Bank's gross NPA ratio in FY25?")
print(f"  Retrieved {len(hyde_results['documents'][0])} chunks")
print(f"  Top result: {hyde_results['documents'][0][0][:150]}...")

print("\n=== Testing Multi-Query (comparative) ===")
mq_results = multi_query_retrieve("Compare capital adequacy ratios across all 5 banks.")
print(f"  Banks covered: {set(c['metadata']['bank_name'] for c in mq_results)}")

print("\n=== Testing Summary Retrieve ===")
sum_results = summary_retrieve("ICICI Bank risk management approach")
print(f"  Retrieved {len(sum_results['documents'][0])} summary nodes")
print(f"  Top result: {sum_results['documents'][0][0][:150]}...")

=== Testing HyDE (factual) ===
  Hypothetical answer: During FY25, HDFC Bank demonstrated strong asset quality with the Gross Non-Performing Assets (GNPA)...
  Retrieved 5 chunks
  Top result: mainly in the SME business Consumer banking segment through Mortgage & working capital (Secured). 
The Bank’s credit deposit ratio stood at 85.54% as ...

=== Testing Multi-Query (comparative) ===
  Paraphrases generated: 3
  Unique chunks after deduplication: 13
  Banks covered: {'SBI', 'HDFC Bank', 'Axis Bank', 'Kotak Mahindra Bank', 'ICICI Bank'}

=== Testing Summary Retrieve ===
  Retrieved 5 summary nodes
  Top result: # ICICI Bank FY25 Annual Report: Master Summary

ICICI Bank delivered strong financial performance in FY25, its 70th year since inception, with standa...


In [ ]:
def grade_context(question, context_text):
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": f"""Given this question: {question}

And this retrieved context: {context_text[:6000]}

Rate how well the context answers the question on a scale of 0.0 to 1.0.
0.0 = context is completely irrelevant.
0.5 = context partially answers the question.
0.8 = context mostly answers the question with minor gaps.
1.0 = context fully answers the question.

Important: if context contains relevant financial data even if incomplete, score at least 0.6.

Respond with only: score|one-sentence rationale
Example: 0.8|Context contains CAR data for most banks."""
        }]
    )
    
    result = response.content[0].text.strip()
    parts = result.split("|")
    score = float(parts[0].strip())
    rationale = parts[1].strip() if len(parts) > 1 else "No rationale"
    return score, rationale


def retrieve_by_type(question, query_type, top_k=5):
    if query_type == "factual":
        results = hyde_retrieve(question, n_results=top_k)
        chunks = []
        for i in range(len(results["documents"][0])):
            chunks.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i]
            })
        return chunks
    
    elif query_type == "comparative":
        return multi_query_retrieve(question, n_results=top_k)
    
    elif query_type == "summary":
        results = summary_retrieve(question)
        chunks = []
        for i in range(len(results["documents"][0])):
            chunks.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i]
            })
        return chunks
    
    else:
        return []


def generate_answer(question, chunks):
    context = "\n\n".join([
        f"[{c['metadata']['bank_name']}, Page {c['metadata'].get('page_number', 'N/A')}, {c['metadata']['level']}]\n{c['text']}"
        for c in chunks
    ])
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": f"""You are a financial analyst assistant. Answer only based on the provided context.
Rules:
1. For every fact or number, cite the source as [Bank Name, Page X].
2. These are FY25 annual reports — assume all figures are FY25 unless stated otherwise.
3. If the context contains relevant data even without explicit FY25 label, use it and cite it.
4. Only say 'Not found in provided reports' if the context has absolutely no relevant information.
5. Never say 'not found' and then quote relevant data in the same response.

Context:
{context}

Question: {question}"""}]
    )
    return response.content[0].text
    

def query_with_grading(question):
    print(f"\nQuestion: {question}")
    
    # step 1 - route
    query_type = classify_query(question)
    print(f"Router: {query_type}")
    
    # step 2 - handle out of scope immediately
    if query_type == "out_of_scope":
        return {
            "answer": "Not found in provided reports.",
            "confidence": 0.0,
            "query_type": query_type,
            "sources": []
        }
    
    # comparative queries need more chunks to cover all 5 banks
    top_k = 10 if query_type == "comparative" else 5
    
    # step 3 - retrieve
    chunks = retrieve_by_type(question, query_type, top_k=top_k)
    context_text = " ".join([c["text"] for c in chunks])
    
    # step 4 - grade
    score, rationale = grade_context(question, context_text)
    print(f"Grade: {score} — {rationale}")
    
    # step 5 - re-retrieve if score too low
    if score < 0.4:
        print(f"Score too low — re-retrieving with higher top_k...")
        top_k = top_k + 8
        chunks = retrieve_by_type(question, query_type, top_k=top_k)
        context_text = " ".join([c["text"] for c in chunks])
        score, rationale = grade_context(question, context_text)
        print(f"New grade: {score} — {rationale}")
    
    # step 6 - return insufficient if still too low
    if score < 0.3:
        return {
            "answer": "Insufficient context found in provided reports.",
            "confidence": score,
            "query_type": query_type,
            "sources": []
        }
    
    # step 7 - generate answer
    answer = generate_answer(question, chunks)
    
    # extract sources
    sources = list({
        (c["metadata"]["bank_name"], c["metadata"].get("page_number", "N/A"))
        for c in chunks
    })
    
    return {
        "answer": answer,
        "confidence": score,
        "query_type": query_type,
        "sources": sources
    }

# test it end to end
test_questions = [
    "What was HDFC Bank's gross NPA ratio in FY25?",
    "Compare capital adequacy ratios across all 5 banks.",
    "What is the RBI repo rate?"
]

for q in test_questions:
    result = query_with_grading(q)
    print(f"\nAnswer: {result['answer'][:600]}")
    print(f"Confidence: {result['confidence']}")
    print(f"Sources: {result['sources']}")
    print("-" * 60)


Question: What was HDFC Bank's gross NPA ratio in FY25?
Router: factual
  Hypothetical answer: During FY25, HDFC Bank maintained robust asset quality with the gross non-performing assets (GNPA) r...
Grade: 1.0 — Context explicitly states HDFC Bank's GNPA ratio was 1.33% in FY25, directly and fully answering the question.

Answer: HDFC Bank's gross NPA ratio in FY25 was **1.33%** [HDFC Bank, Page 38] and [HDFC Bank, Page 15].
Confidence: 1.0
Sources: [('Axis Bank', 240), ('HDFC Bank', 238), ('HDFC Bank', 38), ('HDFC Bank', 15), ('Kotak Mahindra Bank', 408)]
------------------------------------------------------------

Question: Compare capital adequacy ratios across all 5 banks.
Router: comparative
  Paraphrases generated: 3
  Unique chunks after deduplication: 22
Grade: 0.4 — Context provides detailed capital adequacy ratios for only 2 banks across different periods, but the question asks for comparison across all 5 banks.

Answer: # Capital Adequacy Ratios Comparison Across Banks

##

In [10]:
result = query_with_grading("Compare capital adequacy ratios across all 5 banks.")
print(f"\nFull Answer:\n{result['answer']}")
print(f"\nConfidence: {result['confidence']}")
print(f"Sources: {result['sources']}")


Question: Compare capital adequacy ratios across all 5 banks.
Router: comparative
  Paraphrases generated: 3
  Unique chunks after deduplication: 23
Grade: 0.6 — Context contains detailed capital adequacy ratios for 2 banks (one showing CET1 of 15.94%, Tier 1 of 15.94%, Total CRAR of 16.55%; another showing CET1 of 11.07%, Tier 1 of 12.31%, Total capital of 14.44%) but lacks data for the remaining 3 banks needed for complete comparison.

Full Answer:
# Capital Adequacy Ratio Comparison Across Banks (As of March 31, 2025)

## Total Capital Adequacy Ratio (Total CRAR)

1. **HDFC Bank**: 19.6% [HDFC Bank, Page 208]
2. **Kotak Mahindra Bank**: 22.25% (Total Capital ₹115,597.49 crore / RWAs ₹519,614.54 crore) [Kotak Mahindra Bank, Page 232]
3. **Axis Bank**: 17.07% [Axis Bank, Page 243]
4. **ICICI Bank**: 16.55% [ICICI Bank, Page 173]
5. **SBI**: 14.44% [SBI, Page 348]

## Tier 1 Capital Adequacy Ratio

1. **Kotak Mahindra Bank**: 21.10% [Kotak Mahindra Bank, Page 232]
2. **HDFC Bank**: 17

In [12]:
result = hyde_retrieve("Who is the CEO of SBI?", n_results=5)
for i in range(len(result["documents"][0])):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Bank: {result['metadatas'][0][i]['bank_name']}")
    print(f"Page: {result['metadatas'][0][i]['page_number']}")
    print(f"Text: {result['documents'][0][i][:300]}")

  Hypothetical answer: Under the strategic leadership of Chairman Shri Dinesh Kumar Khara, who assumed office on October 7,...

--- Chunk 1 ---
Bank: SBI
Page: 418
Text: ₹ 70,901 Cr 
Net Profit
	 1.10%  
Return on Assets
	 19.87% 
Return on Equity
The Indian economy is regaining its growth 
momentum driven by recovery in consumption 
demand and overall investment. State Bank 
of India demonstrated remarkable resilience, 
delivering a revenue of ₹5,24,172 Crore and 


--- Chunk 2 ---
Bank: SBI
Page: 418
Text: is firmly positioned for facilitating a stronger 
economic 
growth 
and 
financial 
stability, 
guided by a commitment to excellence & 
culture of adaptability.
Economic Value Generated, 
Distributed and Retained
In FY 2024-25, SBI maintained a strong Capital 
Adequacy Ratio of 14.25% demonstrating 

--- Chunk 3 ---
Bank: Axis Bank
Page: 70
Text: Axis Bank delivered a strong performance in fiscal 2025, 
marked by healthy growth across core operating metrics, 
disciplined cost manag